# Módulo 3: Agentes de IA
- **Sesión:** 01 — Fundamentos de agentes de IA
- **Tema:** Patrones de razonamiento  — ReAct y Plan-and-Execute
- **Fecha:** 20/10/2026

## Fundamentos — Patrones de razonamiento

| Patrón | Cuándo decide | Ciclo |
|---|---|---|
| **ReAct** | Durante la tarea, un paso a la vez | razonar → actuar → observar |
| **Plan-and-Execute** | Antes de empezar: escribe el plan completo | planificar → ejecutar → sintetizar |

In [1]:
from langchain.tools import tool
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv
import os

In [2]:

# Generar una funcion que obtenga el modelo LLM

def get_llm(prefix_model, temperature=0.9):
    """
    Obtiene un modelo de lenguaje (LLM) basado en el modelo especificado.

    Args:
        model (str): El nombre del modelo a utilizar.
        temperature (float, optional): La temperatura para la generación de texto. 
                                        Valores más altos producen respuestas más creativas.
                                        Valores más bajos producen respuestas más deterministas.
                                        Por defecto es 0.9.

    Returns:
        llm: Un objeto de modelo de lenguaje inicializado.
    """
    llm = init_chat_model(
                            model=prefix_model, 
                            # temperature=temperature
                          )
    return llm

# Configuración del modelo de lenguaje
load_dotenv(override = True)

prefix_model = os.getenv("PREFIX_MODEL")

# Obtención del modelo
llm = get_llm(prefix_model)



In [3]:
from langchain.tools import tool

# Datos simulados de TiendaYa

## Estado, Precio, Costo de Envio
PEDIDOS = {
    "48213": {"estado": "entregado", "monto": 320.00, "envio": 15.00},
    "77100": {"estado": "en ruta", "monto": 500.00, "envio": 10.00},
    "50555": {"estado": "en preparación", "monto": 1250.00, "envio": 0.00},
}

POLITICAS = {
    "reembolsos": ("El reembolso se abona al mismo medio de pago usado en la compra y tarda de 5 a 10 días hábiles. "
                   "El envío se devuelve solo si el producto llegó dañado."),
    "devoluciones": "15 días calendario desde la entrega; 7 días para productos electrónicos.",
}


@tool
def consultar_pedido(numero_pedido: str) -> str:
    """
        Devuelve el estado, el monto de los productos y el costo de envío de un pedido de TiendaYa.

    """
    p = PEDIDOS.get(numero_pedido)
    if p is None:
        return f"No existe el pedido {numero_pedido}."

    return f"Pedido {numero_pedido}: {p['estado']}, productos S/ {p['monto']:.2f}, envío S/ {p['envio']:.2f}."


@tool
def consultar_politica(tema: str) -> str:
    """
        Devuelve la política de TiendaYa sobre un tema: 'devoluciones' o 'reembolsos' .
    """
    return POLITICAS.get(tema.lower(), "No existe una política sobre ese tema.")


@tool
def calcular_reembolso(monto_productos: float, costo_envio: float, incluir_envio: bool) -> str:
    """
        Calcula el monto a reembolsar. Suma el envío solo si incluir_envio es verdadero.
    """
    
    total = monto_productos + (costo_envio if incluir_envio else 0)
    
    return f"Monto a reembolsar: S/ {total:.2f}"


print(consultar_pedido.invoke(
                                {
                                    "numero_pedido": "48213"
                                }
                            ))

print(consultar_politica.invoke({"tema": "reembolsos"}))

print(calcular_reembolso.invoke({"monto_productos": 320, "costo_envio": 15, "incluir_envio": True}))




Pedido 48213: entregado, productos S/ 320.00, envío S/ 15.00.
El reembolso se abona al mismo medio de pago usado en la compra y tarda de 5 a 10 días hábiles. El envío se devuelve solo si el producto llegó dañado.
Monto a reembolsar: S/ 335.00


## 1 — ReAct: razonar, actuar, observar


In [4]:
CONSULTA = "Mi pedido 48213 llegó dañado. ¿Cuánto me devuelven y en cuántos días?"

from langchain.agents import create_agent
from langchain_core.messages import AIMessage, ToolMessage

SYSTEM_PROMPT="Eres el asistente de TiendaYa. Usa las herramientas para obtener los datos; no los inventes.",

agente = create_agent(
    model=llm,
    tools=[ consultar_pedido, 
            consultar_politica,
            calcular_reembolso],
    system_prompt=SYSTEM_PROMPT
)

# Mensaje
mensaje = {
            "messages": [{
                            "role": "user", 
                            "content": CONSULTA}]
        }

resultado = agente.invoke(mensaje)

print ("------------------------")

'''
for m in resultado["messages"]:
    if getattr(m, "tool_calls", None):
        contenido = "solicita " + ", ".join(f"{t['name']}({t['args']})" for t in m.tool_calls)
    else:
        contenido = m.text
    print(f"[{type(m).__name__:12}] {contenido[:150]}")
#'''

print ("------------------------")

ite = 0
for m in resultado["messages"]:
    if isinstance(m,AIMessage) and m.tool_calls :
        ite += 1
        for tool in m.tool_calls:
            print(f"Ite {ite} | ACTUAR -> {tool['name']}({tool['args']}")
    elif isinstance(m,ToolMessage) :
        print( f"Ite {ite} | OBSERVAR -> {m.content}")

print(resultado["messages"][-1].text)

------------------------
------------------------
Ite 1 | ACTUAR -> consultar_pedido({'numero_pedido': '48213'}
Ite 1 | ACTUAR -> consultar_politica({'tema': 'reembolsos'}
Ite 1 | ACTUAR -> consultar_politica({'tema': 'devoluciones'}
Ite 1 | OBSERVAR -> Pedido 48213: entregado, productos S/ 320.00, envío S/ 15.00.
Ite 1 | OBSERVAR -> El reembolso se abona al mismo medio de pago usado en la compra y tarda de 5 a 10 días hábiles. El envío se devuelve solo si el producto llegó dañado.
Ite 1 | OBSERVAR -> 15 días calendario desde la entrega; 7 días para productos electrónicos.
Ite 2 | ACTUAR -> calcular_reembolso({'incluir_envio': True, 'costo_envio': 15, 'monto_productos': 320}
Ite 2 | OBSERVAR -> Monto a reembolsar: S/ 335.00
Como tu pedido llegó dañado, se te reembolsará el costo total de los productos más el envío, lo que hace un total de **S/ 335.00**.

El dinero será abonado al mismo medio de pago que usaste en tu compra y tardará de **5 a 10 días hábiles** en reflejarse. Además, rec

## 2 — Plan-and-Execute: primero el plan, luego la ejecución

![Plan-and-Execute](graficos/M3_S01_razonamiento_paso2_plan_and_execute.png)


In [5]:
# 

from langchain.agents import create_agent
from langchain_core.messages import AIMessage, ToolMessage

SYSTEM_PROMPT="Eres el asistente de TiendaYa. Usa las herramientas para obtener los datos; no los inventes.",
HERRAMIENTAS = [ consultar_pedido, 
            consultar_politica,
            calcular_reembolso]

agente = create_agent(
    model=llm,
    tools=HERRAMIENTAS,
    system_prompt=SYSTEM_PROMPT
)

# Mensaje
mensaje = {
            "messages": [{
                            "role": "user", 
                            "content": CONSULTA}]
        }

In [6]:
CONSULTA = "Mi pedido 48213 llegó dañado. ¿Cuánto me devuelven y en cuántos días?"

# 1. Planificar

NOMBRES = [h.name for h in HERRAMIENTAS]
DESCRIPCION = "\n".join(f"- {h.name}: {h.description}" for h in HERRAMIENTAS)

# print(DESCRIPCION)

PROMPT_PLAN = f"""

    Herramientas disponibles:
       {DESCRIPCION}
    
    Escribe un plan para atender la consulta del cliente. Cada paso debe usar UNA de las herramientas
    e indicar su nombre. No incluyas pasos que las herramientas no puedan realizar.
    Responde solo con una lista numerada, un paso por línea

    Consulta: {CONSULTA}

"""

#print(PROMPT_PLAN)

texto = llm.invoke(PROMPT_PLAN).text
#print(texto)
plan = [linea.strip() for linea in texto.splitlines() if linea.strip()[:1].isdigit()]


print("PLAN:")
for paso in plan:
    print("  ", paso)


# TO DO

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


PLAN:
   1. consultar_pedido(48213)
   2. consultar_politica('reembolsos')
   3. calcular_reembolso()
